# Group Formation & Optimization

Constrained optimization for complementary peer study groups.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
data_path = os.path.join("..", "..", "data_pipeline", "data", "students.json")
with open(data_path, "r") as f:
    students_raw = json.load(f)

df = pd.json_normalize(students_raw)

subjects = [
    "DSA", "OOP", "DBMS", "OS", "Networks",
    "Mathematics", "Physics", "English", "Machine Learning", "Web Development"
]
slots = ["morning", "afternoon", "evening"]
pace_map = {"slow": 0.0, "moderate": 0.5, "fast": 1.0}

skill_matrix = np.zeros((len(students_raw), len(subjects)))
avail_matrix = np.zeros((len(students_raw), 21))

feature_rows = []
for i, student in enumerate(students_raw):
    row = [student["cgpa"] / 10.0, student["year"] / 4.0, pace_map[student["learning_pace"]]]
    skill_lookup = {s["subject"]: s for s in student["skills"]}
    for j, subj in enumerate(subjects):
        s = skill_lookup.get(subj, {})
        sr = s.get("self_rating", 5.0) / 10.0
        pr = s.get("peer_rating", 5.0) / 10.0
        gp = s.get("grade_points", 5.0) / 10.0
        row.extend([sr, pr, gp])
        skill_matrix[i, j] = (sr + pr + gp) / 3.0
    row.extend([np.mean(skill_matrix[i]), np.std(skill_matrix[i])])
    for entry in student["availability"]:
        bit = entry["day_of_week"] * 3 + slots.index(entry["slot"])
        val = 1.0 if entry["is_available"] else 0.0
        avail_matrix[i, bit] = val
        row.append(val)
    feature_rows.append(row)

X = np.array(feature_rows)
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=8, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)
df["cluster"] = cluster_labels

print(f"Students: {len(df)}, Features: {X_scaled.shape[1]}, Clusters: 8")
print(f"Silhouette score: {silhouette_score(X_scaled, cluster_labels):.4f}")

## 1. Complementary Score Calculation

In [ ]:
def complementary_score(member_indices, skill_matrix, avail_matrix, students_raw, pace_map):
    """Compute complementary score for a group of students.

    Components:
    - skill_diversity: std of skill means across members (higher = more diverse)
    - schedule_overlap: fraction of 21 slots where ALL members are available
    - goal_alignment: Jaccard similarity of goal keywords across members
    - pace_compatibility: 1 - std of pace values (higher = more compatible)
    """
    n = len(member_indices)
    skills = skill_matrix[member_indices]

    member_skill_means = skills.mean(axis=1)
    skill_diversity = float(np.std(member_skill_means))

    avails = avail_matrix[member_indices]
    overlap = np.all(avails == 1, axis=0).sum()
    schedule_overlap = overlap / 21.0

    all_keywords = []
    for idx in member_indices:
        words = set(students_raw[idx]["goals"].lower().split())
        all_keywords.append(words)
    if n > 1:
        intersection = all_keywords[0]
        union = all_keywords[0]
        for kw in all_keywords[1:]:
            intersection = intersection & kw
            union = union | kw
        goal_alignment = len(intersection) / max(len(union), 1)
    else:
        goal_alignment = 1.0

    paces = [pace_map[students_raw[idx]["learning_pace"]] for idx in member_indices]
    pace_compatibility = 1.0 - float(np.std(paces))

    score = (
        0.25 * skill_diversity +
        0.35 * schedule_overlap +
        0.20 * goal_alignment +
        0.20 * pace_compatibility
    )

    return {
        "total_score": round(score, 4),
        "skill_diversity": round(skill_diversity, 4),
        "schedule_overlap": round(schedule_overlap, 4),
        "goal_alignment": round(goal_alignment, 4),
        "pace_compatibility": round(pace_compatibility, 4),
        "overlapping_slots": int(overlap),
    }


sample_indices = list(range(5))
sample_score = complementary_score(sample_indices, skill_matrix, avail_matrix, students_raw, pace_map)

print("Sample group (first 5 students):")
for idx in sample_indices:
    print(f"  {students_raw[idx]['name']} | CGPA={students_raw[idx]['cgpa']} | Pace={students_raw[idx]['learning_pace']}")

print(f"\nComplementary Score Breakdown:")
for key, val in sample_score.items():
    print(f"  {key}: {val}")

## 2. Group Formation Algorithm

In [ ]:
def form_groups(df, skill_matrix, avail_matrix, students_raw, pace_map,
                min_size=4, max_size=6, min_overlap=2):
    """Greedy group formation within clusters.

    For each cluster, greedily build groups by adding the student
    that maximizes the complementary score while maintaining
    minimum schedule overlap.
    """
    groups = []

    for cluster_id in sorted(df["cluster"].unique()):
        members = df[df["cluster"] == cluster_id].index.tolist()
        np.random.seed(42 + cluster_id)
        np.random.shuffle(members)
        remaining = list(members)

        while len(remaining) >= min_size:
            seed = remaining.pop(0)
            group = [seed]

            candidates = list(remaining)
            while len(group) < max_size and candidates:
                best_idx = None
                best_score = -1

                for cand in candidates:
                    trial = group + [cand]
                    trial_avails = avail_matrix[trial]
                    overlap_count = np.all(trial_avails == 1, axis=0).sum()
                    if overlap_count < min_overlap:
                        continue
                    sc = complementary_score(trial, skill_matrix, avail_matrix, students_raw, pace_map)
                    if sc["total_score"] > best_score:
                        best_score = sc["total_score"]
                        best_idx = cand

                if best_idx is not None:
                    group.append(best_idx)
                    candidates.remove(best_idx)
                else:
                    break

            if len(group) >= min_size:
                for m in group[1:]:
                    if m in remaining:
                        remaining.remove(m)
                score = complementary_score(group, skill_matrix, avail_matrix, students_raw, pace_map)
                groups.append({"members": group, "cluster": cluster_id, **score})
            else:
                remaining.extend(group[1:])

        if remaining and groups:
            for leftover in remaining:
                cluster_groups = [g for g in groups if g["cluster"] == cluster_id]
                if cluster_groups:
                    best_g = min(cluster_groups, key=lambda g: len(g["members"]))
                    if len(best_g["members"]) < max_size + 1:
                        best_g["members"].append(leftover)

    return groups


groups = form_groups(df, skill_matrix, avail_matrix, students_raw, pace_map)

print(f"Total groups formed: {len(groups)}")
total_assigned = sum(len(g["members"]) for g in groups)
print(f"Students assigned: {total_assigned} / {len(df)}")
print(f"\nFirst 5 groups:")
for i, g in enumerate(groups[:5]):
    names = [students_raw[m]["name"] for m in g["members"]]
    print(f"  Group {i+1}: {len(g['members'])} members | score={g['total_score']:.4f} | overlap={g['overlapping_slots']} slots")
    print(f"    Members: {', '.join(names)}")

## 3. Group Quality Analysis

In [ ]:
group_sizes = [len(g["members"]) for g in groups]
group_scores = [g["total_score"] for g in groups]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(group_sizes, bins=range(min(group_sizes), max(group_sizes) + 2),
             edgecolor="black", alpha=0.7, color="steelblue", align="left")
axes[0].set_xlabel("Group Size", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title("Distribution of Group Sizes", fontsize=14, fontweight="bold")
axes[0].set_xticks(range(min(group_sizes), max(group_sizes) + 1))

axes[1].hist(group_scores, bins=20, edgecolor="black", alpha=0.7, color="coral")
axes[1].axvline(np.mean(group_scores), color="red", linestyle="--",
                label=f"Mean={np.mean(group_scores):.3f}")
axes[1].set_xlabel("Complementary Score", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("Distribution of Complementary Scores", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"Group size  — min: {min(group_sizes)}, max: {max(group_sizes)}, mean: {np.mean(group_sizes):.1f}")
print(f"Comp. score — min: {min(group_scores):.4f}, max: {max(group_scores):.4f}, mean: {np.mean(group_scores):.4f}")

## 4. Skill Distribution Within Groups

In [ ]:
num_show = min(3, len(groups))
fig, axes = plt.subplots(1, num_show, figsize=(6 * num_show, 6), subplot_kw={"polar": True})
if num_show == 1:
    axes = [axes]

angles = np.linspace(0, 2 * np.pi, len(subjects), endpoint=False).tolist()
angles += angles[:1]

for idx in range(num_show):
    g = groups[idx]
    member_skills = skill_matrix[g["members"]]
    avg_skills = member_skills.mean(axis=0).tolist()
    avg_skills += avg_skills[:1]

    axes[idx].plot(angles, avg_skills, "o-", linewidth=2, color="steelblue")
    axes[idx].fill(angles, avg_skills, alpha=0.25, color="steelblue")
    axes[idx].set_xticks(angles[:-1])
    axes[idx].set_xticklabels(subjects, fontsize=7)
    axes[idx].set_ylim(0, 1)
    axes[idx].set_title(f"Group {idx+1} (n={len(g['members'])})", fontsize=12, fontweight="bold", pad=20)

fig.suptitle("Average Skill Profiles per Group (Radar Chart)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5. Schedule Overlap Visualization

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
slot_names = ["Morning", "Afternoon", "Evening"]

num_show = min(3, len(groups))
fig, axes = plt.subplots(1, num_show, figsize=(6 * num_show, 5))
if num_show == 1:
    axes = [axes]

for idx in range(num_show):
    g = groups[idx]
    member_avails = avail_matrix[g["members"]]
    overlap = np.all(member_avails == 1, axis=0).astype(float)
    overlap_grid = overlap.reshape(7, 3)

    overlap_df = pd.DataFrame(overlap_grid, index=days, columns=slot_names)
    sns.heatmap(
        overlap_df, annot=True, fmt=".0f", cmap="Greens",
        vmin=0, vmax=1, ax=axes[idx], cbar=False,
        linewidths=1, linecolor="gray"
    )
    n_overlap = int(overlap.sum())
    axes[idx].set_title(f"Group {idx+1} Overlap ({n_overlap}/21 slots)", fontsize=12, fontweight="bold")
    axes[idx].set_ylabel("Day")
    axes[idx].set_xlabel("Slot")

fig.suptitle("Schedule Overlap Heatmaps (1 = all members available)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

Groups formed with constrained optimization maximizing complementarity.

**Approach:**
1. Students clustered into 8 groups via KMeans on 63-dim feature vectors
2. Within each cluster, greedy group formation maximizing complementary score
3. Constraints enforced: group size 4-6, minimum 2 overlapping schedule slots

**Complementary score components:**
- Skill diversity (25%): higher variance = more complementary expertise
- Schedule overlap (35%): fraction of slots where all members are free
- Goal alignment (20%): Jaccard similarity of study goal keywords
- Pace compatibility (20%): similarity in learning pace preferences